In [ ]:
### 신경망 모델 학습 - pytorch version

## 패션 MNIST 호출 + 훈련/검증 세트 추출

from torchvision.datasets import FashionMNIST
import torch

fm_train = FashionMNIST(root='.', train=True, download=True)
fm_test =  FashionMNIST(root='.', train=False, download=True)

# Fix: Use .data and .targets attributes directly.
# Also convert data to float for scaling.
train_input = fm_train.data.float()
train_target = fm_train.targets # Targets are usually LongTensor for classification

train_scaled = train_input / 255.0

from sklearn.model_selection import train_test_split

# train_test_split can handle torch.Tensor and will return tensors.
train_scaled, val_scaled, train_target, val_target = train_test_split(
    train_scaled, train_target, test_size=0.2, random_state=42)



## 밀집층 사이 드롭아웃(Dropout) 층 추가 + GPU 적재

import torch.nn as nn

model = nn.Sequential(
    nn.Flatten(),
    nn.Linear(784, 100), ## Keras의 Dense와 같은 역할
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(100, 10) # 출력층(2번째 밀집층) 활성화 함수 생략 - criterion 변수에 포함(nn.CrossEntropyLoss)
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)


## 손실 함수 & 옵티마이저 세팅
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

# 훈련에 필요한 변수/리스트 세팅
train_hist = []
val_hist = []
patience = 2
best_loss = -1
early_stopping_counter = 0

# Fix: Move validation data to device once outside the loop for efficiency
val_scaled = val_scaled.to(device)
val_target = val_target.to(device)


## 파이토치 심층 신경망 훈련 알고리즘 구현(fit() 메서드 X, 직접 코딩)

epochs = 20
batches = int(len(train_scaled) / 32)

for epoch in range(epochs): # 에포크 반복
  model.train() # 드롭아웃 등 훈련-테스트 시에 다르게 작동하는 층 대비 train 지정
  train_loss = 0 # 에포크 손실 초기화

  for batch in range(batches): # 배치 반복
    inputs = train_scaled[batch * 32:(batch + 1)*32].to(device)
    targets = train_target[batch * 32:(batch + 1)*32].to(device) # 배치 입력&타깃 준비
    optimizer.zero_grad() # 옵티마이저 그레디언트 초기화
    outputs = model(inputs) # 입력 전달 to 모델
    loss = criterion(outputs, targets) # 모델 출력 & 타깃에 따라 손실 계산
    loss.backward() # 손실 역전파
    optimizer.step() # 모델 파라미터 업데이트
    train_loss += loss.item() # 에포크 손실 기록(추가)

  model.eval() # 모델, 훈련 이후 검증 모드 돌입
  val_loss = 0
  with torch.no_grad(): # 검증 손실 계산
    # val_scaled and val_target are already on the device from above
    outputs = model(val_scaled)
    loss = criterion(outputs, val_target)
    val_loss = loss.item()

  # 훈련, 검증 손실 리스트에 추가
  train_hist.append(train_loss / batches)
  val_hist.append(val_loss)
  print(f"에포크:{epoch+1},", f"훈련 손실:{train_loss/batches:.4f},", f"검증 손실:{val_loss:.4f}")

  if best_loss == -1 or val_loss < best_loss:
    best_loss = val_loss
    early_stopping_counter = 0
    torch.save(model.state_dict(), 'best_model.pt')
  else:
    early_stopping_counter += 1

  if early_stopping_counter >= patience:
    print(f"{epoch + 1}번째 에포크에서 조기 종료되었습니다.")
    break


# 훈련 & 검증 손실 그래프로 시각화
import matplotlib.pyplot as plt
plt.plot(train_hist, label = 'train')
plt.plot(val_hist, label = 'val')
plt.xlabel('epoch')
plt.ylabel('loss')
plt.legend()
plt.show()

#torch.save로 저장된 최적 파라미터를 torch.load로 불러오기
model.load_state_dict(torch.load('best_model.pt', weights_only = True))

# 검증 세트에 대한 성능 확인
model.eval()
with torch.no_grad():
  val_scaled = val_scaled.to(device)
  val_target = val_target.to(device)
  outputs = model(val_scaled)
  predicts = torch.argmax(outputs, 1)
  corrects = (predicts == val_target).sum().item()

accuracy = corrects / len(val_target)
print(f"검증 세트 정확도: {accuracy:.4f}")

